# 17.13 保住大值與保住小值，如何看取捨？


四個特徵`[0.1,0.2,0.3,100]`共用一個4-bit刻度。為了容納100，刻度是100/7≈14.2857，前三項全落到碼0。這種遠大於其他值的少數數字叫**極端值（outlier）**。

先把來源截到−1至1，刻度變成1/7，可以保住小值的更多細節；但100變成1，差了99。要看這項取捨，誤差必須對原本的x計算，不能只和截斷後的來源比。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.quantization import quantize_symmetric

x = torch.tensor([0.1, 0.2, 0.3, 100.0])
for clip in (False, True):
    source = x.clamp(-1, 1) if clip else x
    q, scale = quantize_symmetric(source, bits=4)
    restored = q.float() * scale
    error = (x - restored).abs()
    print("截斷", clip, "還原值", restored.round(decimals=4).tolist())
    print(
        "小值MAE",
        round(error[:3].mean().item(), 4),
        "極值差",
        round(error[-1].item(), 4),
        "全部MAE",
        round(error.mean().item(), 4),
    )

`x.clamp(-1, 1) if clip else x`是[17.7](https://birdhackor.github.io/tiny-perceptron-vlm/17.7.html)看過的條件寫法：clip為真時用截斷後的版本。關鍵是`error = (x - restored).abs()`，永遠拿原本的x來比。`error[:3]`取前三項，也就是三個小值；`error[-1]`取最後一項，也就是極端值。方括號裡的冒號表示取一段，負數索引從尾端數起。

不截斷時還原值是`[0,0,0,100]`，小值MAE 0.2、極值差0、全部MAE 0.15；截斷後小值MAE約0.0381，極值差99，全部MAE約24.7786。

把全部的平均與「小值、極端值」分開看，結論不同：小值明顯改善，總平均卻變糟。極端值若只是雜訊，截斷可能划算；若它代表任務的重要訊號，就不能丟。本例沒有任務模型，無法替我們決定。

也可以把共用刻度的組切小（[17.5](https://birdhackor.github.io/tiny-perceptron-vlm/17.5.html)的分組量化），或讓敏感的位置保留較高精度；成本是多存刻度、多做運算。截斷界線設多少、哪些位置保留較高精度，這些設定先用範圍校準資料與驗證資料比較，最後才用沒參與挑選的測試資料檢查完整用途。

練習把程式的`x = torch.tensor([0.1, 0.2, 0.3, 100.0])`裡的`100.0`改成`2.0`。先預測：截斷後極值差變成1（2被截到1），三個小值的還原值不變；不截斷時刻度變成2/7，小值不再全部歸零。這次改的是極端值，不是位元數或截斷界線。
